In [ ]:
import os
import pandas as pd
import nibabel as nib

# === Chemins par groupe ===
dossiers = {
    "Rat": "/home/amenacer/Stage/Data/Segmentation Emilien/",
    "CardiaTeam": "/home/amenacer/Stage/Data/nnUnet/nnUNet_raw/Dataset007_MICE_HEART_2D/",
    "Tollsome3D_Groupe1": "/home/amenacer/Stage/Data/TollSome-Segmentation /Segmentation/Tollsome/Groupe1/",
    "Tollsome3D_Groupe2": "/home/amenacer/Stage/Data/TollSome-Segmentation /Segmentation/Tollsome/Groupe2/",
    "Tollsome3D_Groupe3": "/home/amenacer/Stage/Data/TollSome-Segmentation /Segmentation/Tollsome/Groupe3/",
    "Tollsome4D_F1": "/home/amenacer/Stage/data-4D/hadia/TollSome-nifti/img-filtrer/images_filtrees1/",
    "Tollsome4D_F2": "/home/amenacer/Stage/data-4D/hadia/TollSome-nifti/img-filtrer/images_filtrees2/",
    "Tollsome4D_F3": "/home/amenacer/Stage/data-4D/hadia/TollSome-nifti/img-filtrer/images_filtrees3/"
}

# === Initialisation du tableau
table = []

def extraire_infos(fichier, chemin, groupe):
    try:
        full_path = os.path.join(chemin, fichier)
        img = nib.load(full_path)
        shape = img.shape
        type_image = "3D+T" if len(shape) == 4 else "3D"

        # Détection du patient
        if "CTRL" in fichier:
            patient = "CTRL " + fichier.split("_")[2]
            dossier = "Rat"
        elif "HFS" in fichier:
            patient = "HFS " + fichier.split("_")[2]
            dossier = "Rat"
        elif "MICE" in fichier:
            patient = fichier.split("_")[1]
            dossier = "Fatima"
        elif "TollC" in fichier:
            patient = fichier.split("_")[0]
            dossier = groupe.split("_")[-1]  # Groupe1, etc.
        else:
            patient = "?"
            dossier = "?"

        return {
            "Titre": fichier,
            "Groupe": "Rat" if "Rat" in groupe else "CardiaTeam" if "CardiaTeam" in groupe else "Tollsome",
            "Patient": patient,
            "Dossier": dossier,
            "Type": type_image,
            "Dimensions": str(shape),
            "Chemin source": full_path
        }
    except Exception as e:
        print(f"❌ Erreur avec {fichier} : {e}")
        return None

# === Parcours de tous les dossiers
for groupe, chemin in dossiers.items():
    for fichier in os.listdir(chemin):
        if fichier.endswith(".nii.gz"):
            ligne = extraire_infos(fichier, chemin, groupe)
            if ligne:
                table.append(ligne)

# === Export Excel
df = pd.DataFrame(table)
output_path = "/home/amenacer/Stage/tableau_images_chemins_reels.xlsx"
df.to_excel(output_path, index=False)

print(f"✅ Fichier Excel final créé avec {len(df)} images : {output_path}")


In [ ]:
import os
import pandas as pd

# 📁 Dossier contenant les 142 images utilisées dans nnU-Net
imagestr_dir = "/home/amenacer/Stage/Data/nnUnet/nnUNet_raw/Dataset010_MICE_HEART_2D/imagesTr/"
fichiers_imagestr = set([f for f in os.listdir(imagestr_dir) if f.endswith(".nii.gz")])

# 📄 Fichier Excel généré avec les chemins réels
excel_path = "/home/amenacer/Stage/tableau_images_chemins_reels.xlsx"
df = pd.read_excel(excel_path)

# 🎯 Fichiers listés dans l'Excel
fichiers_excel = set(df["Titre"].tolist())

# 🔍 Détection des fichiers manquants
manquants = fichiers_imagestr - fichiers_excel

print(f"🔢 Nombre d’images dans imagesTr : {len(fichiers_imagestr)}")
print(f"📄 Nombre d’images dans Excel     : {len(fichiers_excel)}")
print(f"❌ Images manquantes dans Excel   : {len(manquants)}")

# 📋 Afficher les fichiers manquants
for f in sorted(manquants):
    print(f)


In [ ]:
import os
import pandas as pd
import nibabel as nib

# 📄 Fichier Excel existant (avec 110 images)
excel_path = "/home/amenacer/Stage/tableau_images_chemins_reels.xlsx"
df = pd.read_excel(excel_path)

# 📁 Dossier des 142 vraies images utilisées
imagestr_dir = "/home/amenacer/Stage/Data/nnUnet/nnUNet_raw/Dataset010_MICE_HEART_2D/imagesTr/"
fichiers_imagestr = set([f for f in os.listdir(imagestr_dir) if f.endswith(".nii.gz")])

# 🔍 Fichiers déjà inclus dans le fichier Excel
fichiers_excel = set(df["Titre"].tolist())

# 📌 Liste des fichiers manquants
manquants = fichiers_imagestr - fichiers_excel

# 📋 Préparation des nouvelles lignes à ajouter
nouvelles_lignes = []

for fichier in sorted(manquants):
    full_path = os.path.join(imagestr_dir, fichier)
    try:
        img = nib.load(full_path)
        shape = img.shape
        type_image = "3D+T" if len(shape) == 4 else "3D"

        if "CTRL" in fichier:
            groupe = "Rat"
            patient = "CTRL " + fichier.split("_")[2]
            dossier = "Rat"
        elif "HFS" in fichier:
            groupe = "Rat"
            patient = "HFS " + fichier.split("_")[2]
            dossier = "Rat"
        elif "MICE-HEART" in fichier:
            groupe = "CardiaTeam"
            patient = fichier.split("_")[1]
            dossier = "Fatima"
        else:
            groupe = "Inconnu"
            patient = "?"
            dossier = "?"

        nouvelles_lignes.append({
            "Titre": fichier,
            "Groupe": groupe,
            "Patient": patient,
            "Dossier": dossier,
            "Type": type_image,
            "Dimensions": str(shape),
            "Chemin source": full_path
        })

    except Exception as e:
        print(f"❌ Erreur lecture {fichier} : {e}")

# ✅ Fusion avec l'ancien tableau
df_complet = pd.concat([df, pd.DataFrame(nouvelles_lignes)], ignore_index=True)

# 💾 Export du nouveau tableau Excel complet
output_path = "/home/amenacer/Stage/tableau_images_complet_corrigé.xlsx"
df_complet.to_excel(output_path, index=False)

print(f"✅ Nouveau fichier Excel enregistré avec {len(df_complet)} images : {output_path}")
